# Run NMME EMOV Processing

This notebook drives `scripts/run_process_modes_of_variability.py` for NMME gridded SST modes of variability.

It mirrors the model selection and archive checks from `preprocessing/sst/0_run_nmme_sst_index.ipynb`, but processes full gridded fields for projected EOF mode diagnostics. NMME SST modes use `sst`; pressure modes use NMME `prmsl`.

Primary outputs are written below:

`/global/cfs/cdirs/e3sm/S2S2D/s2d_diag/NMME/modes_variability/`


In [1]:
import json
import os
import re
import subprocess
import sys
from pathlib import Path

import pandas as pd
import xarray as xr

def find_repo_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "scripts" / "run_process_modes_of_variability.py").is_file():
            return candidate
    raise FileNotFoundError("Could not locate the ESP-Lab repository root.")

REPO_ROOT = find_repo_root()

SCRIPT_PATH = REPO_ROOT / "scripts" / "run_process_modes_of_variability.py"
print(f"Repository root: {REPO_ROOT}")
print(f"Python         : {sys.executable}")
print(f"Script         : {SCRIPT_PATH}")


Repository root: /global/u2/z/zhan391/code/ESP-Lab
Python         : /global/homes/z/zhan391/.conda/envs/e3sm_analysis/bin/python
Script         : /global/u2/z/zhan391/code/ESP-Lab/scripts/run_process_modes_of_variability.py


## Configuration

Set `MODE_GROUP` to `"sst"` or `"pressure"` and run the groups separately to limit memory use. The separate runs merge into the same manifest. Leave `MODELS = []` to use `MODEL_SET`, or keep the explicit list below to match the NMME SST-index preprocessing notebook.


In [2]:
NMME_ROOT = Path("/global/cfs/cdirs/e3sm/S2S2D/NMME/data_hindcast_by_member")
OUTDIR = Path("/global/cfs/cdirs/e3sm/S2S2D/s2d_diag")
NMME_FIXED_DIR = Path("/global/cfs/cdirs/e3sm/S2S2D/s2d_diag/NMME/fixed")
MODEL_SET = "all"  # Used only when MODELS is empty: "all" or "yeager-f03".

MODE_GROUPS = {
    "pressure": [
        "NAM", "NAO", "SAM", "PSA1", "PSA2",
        "PNA", "NPO", "EA", "SCA",
    ],
    "sst": ["PDO", "NPGO", "AMO"],
}
MODE_GROUP = "sst" #"sst"  # Run "pressure" and "sst" as separate jobs.
if MODE_GROUP not in MODE_GROUPS:
    raise ValueError(f"Unknown MODE_GROUP={MODE_GROUP!r}; choose from {list(MODE_GROUPS)}")
MODES = MODE_GROUPS[MODE_GROUP]

INIT_MONTHS = [2, 5, 8, 11]
TARGET_DLAT = 2.5
TARGET_DLON = 2.5
DASK_WORKERS = 2
NMME_CHUNKS = ""  # Use the archive-native chunks (S:12,L:12,Y:45,X:72).
NMME_SST_LAND_MASK = True
FORCE = False
DRY_RUN = False
RUN_DRY_RUN_FIRST = False #True

# Period selection:
#   "standard": require the configured 1981-2010 climatology and retain
#               only models spanning it.
#   "common": use the overlap across all candidate models for both the
#             processing period and climatology (currently 1991-2009).
PERIOD_MODE = "standard"

# NMME SST files typically provide 12 monthly leads.
START_YEAR = "common"  # Use "common" for selected-model overlap, or set an integer year.
END_YEAR = "common"    # Use "common" for selected-model overlap, or set an integer year.
CLIM_START = 1981
CLIM_END = 2010
MONTHLY_NLEAD = 12

OBS_START_YEAR = 1950
OBS_END_YEAR = 2020
EOF_REFERENCE_START_YEAR = 1950
EOF_REFERENCE_END_YEAR = 2020


MODELS = [
    "CanSIPS-IC3",
    "GFDL-CM2p1",
    "NASA-GMAO-062012",
    "CanSIPS-IC4",
    "GFDL-CM2p1-aer04",
    "NCAR-CESM1",
    "CanSIPSv2",
    "GFDL-CM2p5-FLOR-A06",
    "NCEP-CFSv1",
    "CMC1-CanCM3",
    "GFDL-CM2p5-FLOR-B01",
    "NCEP-CFSv2",
    "CMC2-CanCM4",
    "GFDL-SPEAR",
    "GEM-NEMO",
    "NASA-GMAO",
    "CanCM4i",
    "NASA-GEOSS2S",
    "COLA-RSMAS-CCSM3",
    "IRI-ECHAM4p5-AnomalyCoupled",
    "COLA-RSMAS-CCSM4",
    "IRI-ECHAM4p5-DirectCoupled",
    "COLA-RSMAS-CESM1"
]

YEAGER_F03_MODELS = [
    "CanCM4i",
    "COLA-RSMAS-CCSM4",
    "GEM-NEMO",
    "GFDL-CM2p5-FLOR-B01",
    "NASA-GMAO",
    "NCEP-CFSv2",
    "CanSIPSv2",
    "GFDL-SPEAR",
]


## Validate Inputs

This cell follows the NMME SST-index notebook's archive checks, including explicit model selection and rough initialization-year coverage.


In [3]:
missing = []
for label, candidate in {
    "script": SCRIPT_PATH,
    "NMME root": NMME_ROOT,
}.items():
    if not candidate.exists():
        missing.append(f"{label}: {candidate}")

if MODEL_SET not in {"all", "yeager-f03"}:
    missing.append(f"MODEL_SET must be 'all' or 'yeager-f03', got {MODEL_SET!r}")

if missing:
    raise FileNotFoundError("Missing or invalid configuration:\n" + "\n".join(missing))

available_models = sorted(
    path.name for path in NMME_ROOT.iterdir()
    if path.is_dir() and path.name != "logs" and (path / "sst").is_dir()
)

EXPLICIT_MODELS = list(
    dict.fromkeys(str(model).strip() for model in MODELS if str(model).strip())
)
selection_errors = []
if EXPLICIT_MODELS:
    unavailable_models = sorted(set(EXPLICIT_MODELS) - set(available_models))
    if unavailable_models:
        selection_errors.append(
            "MODELS contains names that are not available under NMME_ROOT: "
            + ", ".join(unavailable_models)
        )
    SELECTED_MODELS = EXPLICIT_MODELS
    model_selection = "explicit MODELS list"
elif MODEL_SET == "all":
    SELECTED_MODELS = available_models
    model_selection = "MODEL_SET='all'"
else:
    unavailable_models = sorted(set(YEAGER_F03_MODELS) - set(available_models))
    if unavailable_models:
        selection_errors.append(
            "MODEL_SET='yeager-f03' includes unavailable models under NMME_ROOT: "
            + ", ".join(unavailable_models)
        )
    SELECTED_MODELS = YEAGER_F03_MODELS
    model_selection = "MODEL_SET='yeager-f03'"

if selection_errors:
    raise ValueError("Invalid model selection:\n" + "\n".join(selection_errors))

s_chunk_pattern = re.compile(r"_S(\d+)-(\d+)\.nc$")

def _chunk_range(path):
    match = s_chunk_pattern.search(path.name)
    return (int(match.group(1)), int(match.group(2))) if match else None

def _decode_s_edge(path, first=True):
    with xr.open_dataset(path, decode_times=False) as ds:
        s_ds = ds[["S"]].copy()
    if s_ds["S"].attrs.get("calendar") == "360":
        s_ds["S"].attrs["calendar"] = "360_day"
    decoded = xr.decode_cf(s_ds, decode_times=True)["S"]
    return decoded.values[0 if first else -1]

def _model_s_years(model):
    files = []
    for path in (NMME_ROOT / model / "sst").glob("M*/*.nc"):
        chunk_range = _chunk_range(path)
        if chunk_range is not None:
            files.append((*chunk_range, path))
    if not files:
        raise FileNotFoundError(f"No SST chunks found for {model}")
    first_file = min(files, key=lambda item: (item[0], item[2]))[2]
    last_file = max(files, key=lambda item: (item[1], item[2]))[2]
    first = _decode_s_edge(first_file, first=True)
    last = _decode_s_edge(last_file, first=False)
    return int(first.year), int(last.year)

candidate_model_year_ranges = {model: _model_s_years(model) for model in SELECTED_MODELS}
if PERIOD_MODE not in {"standard", "common"}:
    raise ValueError(f"PERIOD_MODE must be 'standard' or 'common', got {PERIOD_MODE!r}")
candidate_common_start = max(start for start, _ in candidate_model_year_ranges.values())
candidate_common_end = min(end for _, end in candidate_model_year_ranges.values())
if candidate_common_start > candidate_common_end:
    raise ValueError("Candidate NMME models have no overlapping year window.")
if PERIOD_MODE == "common":
    CLIM_START = candidate_common_start
    CLIM_END = candidate_common_end

# Use only models whose local archive spans the complete 30-year normal.
# This prevents a nominal 1981-2010 climatology from silently using fewer years.
excluded_for_climatology = {
    model: (first_year, last_year)
    for model, (first_year, last_year) in candidate_model_year_ranges.items()
    if first_year > CLIM_START or last_year < CLIM_END
}
SELECTED_MODELS = [
    model for model in SELECTED_MODELS
    if model not in excluded_for_climatology
]
if not SELECTED_MODELS:
    raise ValueError(
        f"No selected NMME model spans the full {CLIM_START}-{CLIM_END} climatology."
    )
model_year_ranges = {
    model: candidate_model_year_ranges[model]
    for model in SELECTED_MODELS
}
coverage = pd.DataFrame(
    [
        {
            "model": model,
            "first_year": years[0],
            "last_year": years[1],
            "selected": model in SELECTED_MODELS,
            "reason": (
                "full climatology coverage"
                if model in SELECTED_MODELS
                else f"does not span {CLIM_START}-{CLIM_END}"
            ),
        }
        for model, years in candidate_model_year_ranges.items()
    ]
).sort_values(["selected", "model"], ascending=[False, True])

common_start_year = max(start for start, _ in model_year_ranges.values())
common_end_year = min(end for _, end in model_year_ranges.values())
START_YEAR_RESOLVED = common_start_year if START_YEAR == "common" else int(START_YEAR)
END_YEAR_RESOLVED = common_end_year if END_YEAR == "common" else int(END_YEAR)

if START_YEAR_RESOLVED > END_YEAR_RESOLVED:
    raise ValueError(
        f"Resolved START_YEAR ({START_YEAR_RESOLVED}) must be <= END_YEAR ({END_YEAR_RESOLVED})."
    )
if not (START_YEAR_RESOLVED <= CLIM_START <= CLIM_END <= END_YEAR_RESOLVED):
    raise ValueError(
        "CLIM_START/CLIM_END must lie within the resolved NMME processing years: "
        f"{START_YEAR_RESOLVED}-{END_YEAR_RESOLVED}"
    )

year_errors = []
for model, (first_year, last_year) in model_year_ranges.items():
    if first_year > START_YEAR_RESOLVED or last_year < END_YEAR_RESOLVED:
        year_errors.append(
            f"{model}: available {first_year}-{last_year}, requested "
            f"{START_YEAR_RESOLVED}-{END_YEAR_RESOLVED}"
        )
if year_errors:
    print("Some selected models do not fully cover the resolved years:")
    for item in year_errors:
        print(" ", item)

print(f"Period mode: {PERIOD_MODE}")
print(f"Candidate NMME models via {model_selection}: {len(candidate_model_year_ranges)}")
print(f"Eligible models spanning {CLIM_START}-{CLIM_END}: {len(SELECTED_MODELS)}")
if excluded_for_climatology:
    print("Excluded models without the complete climatology window:")
    for model, (first_year, last_year) in excluded_for_climatology.items():
        print(f"  {model}: available {first_year}-{last_year}")
print(f"Common selected-model years: {common_start_year}-{common_end_year}")
print(f"Resolved years: {START_YEAR_RESOLVED}-{END_YEAR_RESOLVED}; climatology: {CLIM_START}-{CLIM_END}")
display(coverage)


Warning 3: Cannot find header.dxf (GDAL_DATA is not defined)


Period mode: standard
Candidate NMME models via explicit MODELS list: 23
Eligible models spanning 1981-2010: 12
Excluded models without the complete climatology window:
  GFDL-CM2p1: available 1982-2012
  CanSIPS-IC4: available 1990-2024
  GFDL-CM2p1-aer04: available 1982-2021
  NCEP-CFSv1: available 1981-2009
  NCEP-CFSv2: available 1982-2010
  GFDL-SPEAR: available 1991-2020
  COLA-RSMAS-CCSM3: available 1982-2018
  IRI-ECHAM4p5-AnomalyCoupled: available 1982-2012
  COLA-RSMAS-CCSM4: available 1982-2026
  IRI-ECHAM4p5-DirectCoupled: available 1982-2012
  COLA-RSMAS-CESM1: available 1982-2026
Common selected-model years: 1981-2010
Resolved years: 1981-2010; climatology: 1981-2010


,model,first_year,last_year,selected,reason
9,CMC1-CanCM3,1981,2010,True,full climatology coverage
12,CMC2-CanCM4,1981,2010,True,full climatology coverage
16,CanCM4i,1981,2018,True,full climatology coverage
0,CanSIPS-IC3,1980,2021,True,full climatology coverage
6,CanSIPSv2,1981,2018,True,full climatology coverage
14,GEM-NEMO,1981,2018,True,full climatology coverage
7,GFDL-CM2p5-FLOR-A06,1980,2021,True,full climatology coverage
10,GFDL-CM2p5-FLOR-B01,1980,2021,True,full climatology coverage
17,NASA-GEOSS2S,1981,2017,True,full climatology coverage
15,NASA-GMAO,1981,2012,True,full climatology coverage


## Build Command


In [4]:
def build_command(dry_run=False):
    command = [
        sys.executable,
        str(SCRIPT_PATH),
        "--outdir", str(OUTDIR),
        "--modes", *MODES,
        "--sources", "obs", "nmme",
        "--init-months", *(str(month) for month in INIT_MONTHS),
        "--start-year", str(START_YEAR_RESOLVED),
        "--end-year", str(END_YEAR_RESOLVED),
        "--clim-start", str(CLIM_START),
        "--clim-end", str(CLIM_END),
        "--obs-start-year", str(OBS_START_YEAR),
        "--obs-end-year", str(OBS_END_YEAR),
        "--eof-reference-start-year", str(EOF_REFERENCE_START_YEAR),
        "--eof-reference-end-year", str(EOF_REFERENCE_END_YEAR),
        "--monthly-nlead", str(MONTHLY_NLEAD),
        "--target-dlat", str(TARGET_DLAT),
        "--target-dlon", str(TARGET_DLON),
        "--dask-workers", str(DASK_WORKERS),
        "--nmme-root", str(NMME_ROOT),
        "--nmme-fixed-dir", str(NMME_FIXED_DIR),
        "--nmme-models", *SELECTED_MODELS,
        "--nmme-field", "auto",
        "--nmme-chunks", NMME_CHUNKS,
        "--nmme-sst-land-mask" if NMME_SST_LAND_MASK else "--no-nmme-sst-land-mask",
        "--ts-obs-product", "HadISST2",
        "--ts-obs-var", "sst",
        "--eof-strategy", "fixed_obs_projection",
        "--eof-bootstrap-iterations", "0",
        "--merge-manifest",
    ]
    if FORCE:
        command.append("--force")
    if dry_run or DRY_RUN:
        command.append("--dry-run")
    return command

cmd = build_command(dry_run=False)
print("Command:")
print(" ".join(map(str, cmd)))


Command:
/global/homes/z/zhan391/.conda/envs/e3sm_analysis/bin/python /global/u2/z/zhan391/code/ESP-Lab/scripts/run_process_modes_of_variability.py --outdir /global/cfs/cdirs/e3sm/S2S2D/s2d_diag --modes PDO NPGO AMO --sources obs nmme --init-months 2 5 8 11 --start-year 1981 --end-year 2010 --clim-start 1981 --clim-end 2010 --obs-start-year 1950 --obs-end-year 2020 --eof-reference-start-year 1950 --eof-reference-end-year 2020 --monthly-nlead 12 --target-dlat 2.5 --target-dlon 2.5 --dask-workers 2 --nmme-root /global/cfs/cdirs/e3sm/S2S2D/NMME/data_hindcast_by_member --nmme-fixed-dir /global/cfs/cdirs/e3sm/S2S2D/s2d_diag/NMME/fixed --nmme-models CanSIPS-IC3 NASA-GMAO-062012 NCAR-CESM1 CanSIPSv2 GFDL-CM2p5-FLOR-A06 CMC1-CanCM3 GFDL-CM2p5-FLOR-B01 CMC2-CanCM4 GEM-NEMO NASA-GMAO CanCM4i NASA-GEOSS2S --nmme-field auto --nmme-chunks  --nmme-sst-land-mask --ts-obs-product HadISST2 --ts-obs-var sst --eof-strategy fixed_obs_projection --eof-bootstrap-iterations 0 --merge-manifest


## Dry Run


In [5]:
if RUN_DRY_RUN_FIRST:
    dry_cmd = build_command(dry_run=True)
    print("Dry-run command:")
    print(" ".join(map(str, dry_cmd)))
    dry_run = subprocess.run(
        dry_cmd,
        cwd=REPO_ROOT,
        text=True,
        capture_output=True,
        check=True,
    )
    print(dry_run.stdout)
    if dry_run.stderr:
        print(dry_run.stderr)
else:
    print("Skipping dry run because RUN_DRY_RUN_FIRST=False.")


Skipping dry run because RUN_DRY_RUN_FIRST=False.


## Run Processing


In [6]:
env = os.environ.copy()
conda_prefix = Path(sys.prefix)
for env_name, relpath in {
    "GDAL_DATA": "share/gdal",
    "PROJ_LIB": "share/proj",
    "PROJ_DATA": "share/proj",
}.items():
    candidate = conda_prefix / relpath
    if candidate.exists():
        env[env_name] = str(candidate)

def run_processing():
    command = build_command(dry_run=DRY_RUN)
    print("Running command:")
    print(" ".join(map(str, command)))
    completed = subprocess.run(
        command,
        cwd=REPO_ROOT,
        env=env,
        text=True,
        check=True,
    )
    return completed

completed = run_processing()
completed.returncode


Running command:
/global/homes/z/zhan391/.conda/envs/e3sm_analysis/bin/python /global/u2/z/zhan391/code/ESP-Lab/scripts/run_process_modes_of_variability.py --outdir /global/cfs/cdirs/e3sm/S2S2D/s2d_diag --modes PDO NPGO AMO --sources obs nmme --init-months 2 5 8 11 --start-year 1981 --end-year 2010 --clim-start 1981 --clim-end 2010 --obs-start-year 1950 --obs-end-year 2020 --eof-reference-start-year 1950 --eof-reference-end-year 2020 --monthly-nlead 12 --target-dlat 2.5 --target-dlon 2.5 --dask-workers 2 --nmme-root /global/cfs/cdirs/e3sm/S2S2D/NMME/data_hindcast_by_member --nmme-fixed-dir /global/cfs/cdirs/e3sm/S2S2D/s2d_diag/NMME/fixed --nmme-models CanSIPS-IC3 NASA-GMAO-062012 NCAR-CESM1 CanSIPSv2 GFDL-CM2p5-FLOR-A06 CMC1-CanCM3 GFDL-CM2p5-FLOR-B01 CMC2-CanCM4 GEM-NEMO NASA-GMAO CanCM4i NASA-GEOSS2S --nmme-field auto --nmme-chunks  --nmme-sst-land-mask --ts-obs-product HadISST2 --ts-obs-var sst --eof-strategy fixed_obs_projection --eof-bootstrap-iterations 0 --merge-manifest


INFO: Modes: PDO, NPGO, AMO
INFO: Sources: obs, nmme
INFO: Output: /global/cfs/cdirs/e3sm/S2S2D/s2d_diag
INFO: EOF strategy: fixed_obs_projection
INFO: EOF reference period: 1950-2020
INFO: PDO HadISST2: using field product /global/cfs/cdirs/e3sm/S2S2D/s2d_diag/HadISST2/modes_variability/fields/hadisst2_sst_monthly_2p5x2p5deg.nc
INFO: Reference SST ocean mask retained 758/1281 grid cells (59.2%).
INFO: Reference SST ocean mask retained 758/1281 grid cells (59.2%).
INFO: Reference SST ocean mask retained 758/1281 grid cells (59.2%).
INFO: Reference SST ocean mask retained 758/1281 grid cells (59.2%).
INFO: Reference SST ocean mask retained 758/1281 grid cells (59.2%).
INFO: Reference SST ocean mask retained 758/1281 grid cells (59.2%).
INFO: Reference SST ocean mask retained 758/1281 grid cells (59.2%).
INFO: Reference SST ocean mask retained 758/1281 grid cells (59.2%).
INFO: Reference SST ocean mask retained 758/1281 grid cells (59.2%).
INFO: Reference SST ocean mask retained 758/1281

CalledProcessError: Command '['/global/homes/z/zhan391/.conda/envs/e3sm_analysis/bin/python', '/global/u2/z/zhan391/code/ESP-Lab/scripts/run_process_modes_of_variability.py', '--outdir', '/global/cfs/cdirs/e3sm/S2S2D/s2d_diag', '--modes', 'PDO', 'NPGO', 'AMO', '--sources', 'obs', 'nmme', '--init-months', '2', '5', '8', '11', '--start-year', '1981', '--end-year', '2010', '--clim-start', '1981', '--clim-end', '2010', '--obs-start-year', '1950', '--obs-end-year', '2020', '--eof-reference-start-year', '1950', '--eof-reference-end-year', '2020', '--monthly-nlead', '12', '--target-dlat', '2.5', '--target-dlon', '2.5', '--dask-workers', '2', '--nmme-root', '/global/cfs/cdirs/e3sm/S2S2D/NMME/data_hindcast_by_member', '--nmme-fixed-dir', '/global/cfs/cdirs/e3sm/S2S2D/s2d_diag/NMME/fixed', '--nmme-models', 'CanSIPS-IC3', 'NASA-GMAO-062012', 'NCAR-CESM1', 'CanSIPSv2', 'GFDL-CM2p5-FLOR-A06', 'CMC1-CanCM3', 'GFDL-CM2p5-FLOR-B01', 'CMC2-CanCM4', 'GEM-NEMO', 'NASA-GMAO', 'CanCM4i', 'NASA-GEOSS2S', '-

## Product Summary


In [ ]:
manifest_path = OUTDIR / "_manifests" / "modes_manifest.json"
legacy_manifest_path = OUTDIR / "modes_manifest.json"
if not manifest_path.exists() and legacy_manifest_path.exists():
    manifest_path = legacy_manifest_path
if not manifest_path.exists():
    print("No manifest found yet:", manifest_path)
else:
    manifest = json.loads(manifest_path.read_text())
    rows = []
    for product, details in manifest.get("products", {}).items():
        mode, source = product.split(":", 1)
        if mode not in MODES or not (source == "reference" or source.startswith("nmme_init")):
            continue
        index_path = Path(details["index"])
        field_path = Path(details["field"])
        rows.append(
            {
                "product": product,
                "status": details.get("status", ""),
                "index_exists": index_path.exists(),
                "field_exists": field_path.exists(),
                "index_MB": round(index_path.stat().st_size / 1e6, 2) if index_path.exists() else None,
                "field_GB": round(field_path.stat().st_size / 1e9, 2) if field_path.exists() else None,
            }
        )
    summary = pd.DataFrame(rows).sort_values("product").reset_index(drop=True)
    display(summary)
    print("Manifest:", manifest_path)
